# Data Storage Architectures: Warehouses, Lakes, and Lakehouses

**BINFX410 — Chapter 10: Data Engineering Foundations**

---

Modern data-driven organizations store and analyze data using three major architectural patterns: **data warehouses**, **data lakes**, and **lakehouses**. Each solves a different problem, and understanding the trade-offs between them is essential for anyone building analytics pipelines or working with large datasets in biology, medicine, or any data-intensive domain.

In this module we use only **local, open-source tools** — no cloud account required.

## Learning Objectives

By the end of this module you will be able to:

1. Explain the core differences between data warehouses, data lakes, and lakehouses.
2. Describe the historical evolution of data storage architectures.
3. Generate and inspect a realistic synthetic genomics dataset.
4. Choose the right architecture for a given analytical workload.
5. Implement all three architectures locally using DuckDB, PyArrow, and Delta Lake.

## 1. Historical Timeline

```
1990s ──────────────── 2010s ──────────────── 2020s
  │                     │                      │
  ▼                     ▼                      ▼
DATA WAREHOUSE        DATA LAKE            LAKEHOUSE
─────────────         ─────────            ─────────
• Structured SQL      • Raw everything     • ACID on object store
• Schema-on-write     • Schema-on-read     • Schema enforcement
• Expensive           • Cheap storage      • Unified compute
• ETL pipelines       • ELT pipelines      • Open table formats
• Oracle, Teradata    • HDFS, S3           • Delta, Iceberg, Hudi
• BI dashboards       • ML / data science  • BI + ML + streaming
```

### Why did each emerge?

| Era | Problem being solved | Solution |
|-----|---------------------|----------|
| 1990s | Business reporting was slow; OLTP databases can't handle analytics | Data Warehouse — separate, optimized analytical store |
| 2010s | Unstructured data (logs, images, genomics) didn't fit warehouse schemas; storage was cheap | Data Lake — dump everything raw, figure out structure later |
| 2020s | Data lakes became "data swamps"; warehouses still can't handle ML workloads natively | Lakehouse — ACID transactions + open formats on cheap object storage |

### Genomics examples by era

| Architecture | Genomics Best For |
|-------------|-------------------|
| Data Warehouse | Genomic variant databases with structured SQL queries and clinical reporting |
| Data Lake | Raw FASTQ/BAM file storage, ad-hoc exploration with custom Python scripts |
| Lakehouse | Variant + phenotype unified analytics — SQL dashboards AND ML pathogenicity models on the same data |

## 2. Architecture Deep-Dives

### 2.1 Data Warehouse

```
  Source Systems          ETL Pipeline         Data Warehouse
  ──────────────          ────────────         ──────────────
  [VCF files   ]          [Extract    ]        [dim_samples  ]
  [BAM files   ] ──────►  [Transform  ] ─────► [dim_genes    ]
  [Lab LIMS    ]          [Load       ]        [fact_variants ]
                                               [fact_calls    ]
                                                     │
                                                     ▼
                                               BI Tools / SQL
```

Key characteristics:
- **Schema-on-write**: data is cleaned and structured *before* loading
- **Star/snowflake schema**: optimized for aggregation queries
- **ACID transactions**: full consistency guarantees
- **High query performance** for known query patterns
- **Expensive** for storing raw/unstructured data

### 2.2 Data Lake

```
  Source Systems          Ingest               Data Lake Zones
  ──────────────          ──────               ───────────────
  [FASTQ reads ]          [Copy as-is]         [Bronze / Raw ]
  [BAM / CRAM  ] ──────►  [Minimal    ] ─────► [Silver/Clean ]
  [VCF files   ]          [transform  ]        [Gold/Curated ]
  [Annotations ]                                     │
                                                     ▼
                                               Spark / Pandas / SQL
```

Key characteristics:
- **Schema-on-read**: structure is applied when you *query*, not when you store
- **Cheap storage**: object storage (S3, local files) is inexpensive
- **Any data type**: structured, semi-structured, unstructured
- **No ACID**: files are immutable once written; updates require full rewrites
- **Data swamp risk**: without governance, becomes impossible to navigate

### 2.3 Lakehouse

```
  Data Lake Storage (cheap object store)
  ──────────────────────────────────────
  [Parquet files            ]
  [_delta_log/ (JSON logs)  ]  ◄── Transaction log = ACID
  [_delta_log/00000.json    ]
  [_delta_log/00001.json    ]
         │
         ▼
  Open Table Format (Delta Lake / Apache Iceberg / Apache Hudi)
         │
         ├──► SQL queries (DuckDB, Spark SQL, Trino)
         ├──► ML workloads (pandas, PyTorch, scikit-learn)
         └──► Streaming (structured streaming)
```

Key characteristics:
- **ACID on object storage**: transaction log provides consistency without a database server
- **Time travel**: query any historical version of the data
- **Schema enforcement + evolution**: catch bad data, evolve safely
- **Open formats**: Parquet files are readable by any tool
- **Unified**: same data serves BI, ML, and streaming workloads

## 3. Architecture Comparison Table

| Feature | Data Warehouse | Data Lake | Lakehouse |
|---------|---------------|-----------|-----------|
| **Schema** | Schema-on-write | Schema-on-read | Schema-on-write + evolution |
| **ACID Transactions** | Yes (full) | No | Yes (via transaction log) |
| **Storage Format** | Proprietary columnar | Any (CSV, JSON, Parquet) | Open (Parquet + log) |
| **Query Performance** | Excellent (known queries) | Poor–Good (depends on format) | Good–Excellent |
| **Storage Cost** | High | Low | Low |
| **Data Types** | Structured only | Any | Structured + semi-structured |
| **ML Workloads** | Poor | Good | Excellent |
| **Streaming** | Difficult | Possible | Native support |
| **Data Governance** | Strong | Weak | Strong |
| **Time Travel** | Limited | No | Yes (built-in) |
| **Best For** | Genomic variant databases | Raw FASTQ/BAM storage | Variant + phenotype unified analytics |
| **Local Tool** | DuckDB | PyArrow + Parquet | Delta Lake + DuckDB |

## 4. Environment Setup

Before running the notebooks, install dependencies:

```bash
pip install -r requirements.txt
```

The cell below verifies your environment.

In [1]:
import sys
import importlib

required = {
    'duckdb': '0.10.0',
    'pandas': '2.0.0',
    'pyarrow': '14.0.0',
    'deltalake': '0.15.0',
    'faker': '20.0.0',
    'matplotlib': '3.7.0',
    'seaborn': '0.12.0',
}

print(f'Python {sys.version}\n')
all_ok = True
for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  OK  {pkg:12s} {ver}')
    except ImportError:
        print(f'  MISSING  {pkg}  (need >= {min_ver})')
        all_ok = False

print()
if all_ok:
    print('All dependencies found. You are ready to go!')
else:
    print('Run:  pip install -r requirements.txt')

Python 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:37:07) [Clang 15.0.7 ]

  OK  duckdb       1.4.3
  OK  pandas       2.3.2
  OK  pyarrow      21.0.0
  OK  deltalake    1.5.0
  OK  faker        unknown
  OK  matplotlib   3.10.6
  OK  seaborn      0.13.2

All dependencies found. You are ready to go!


## 5. Generating the Synthetic Genomics Dataset

We generate a realistic genomics dataset covering sequencing samples, gene annotations, variant calls, and individual variants. This gives us:

- **500 samples** — tissue type, sequencing platform, library prep, coverage metrics, project assignment
- **100 genes** — real and synthetic cancer gene names, chromosomal positions, biotype annotations
- **2,000 variant calls** — linked to samples, with caller and filter status
- **5,000 variants** — individual mutations linked to calls and genes, with allele frequency and quality scores

This dataset mimics what you might see in a cancer genomics sequencing center and will be used across all five notebooks.

We save everything as CSV files in `./raw_data/` — the starting point of all three architectures.

In [2]:
import os
import random
from datetime import date, timedelta

import pandas as pd
from faker import Faker

# Reproducibility
SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

os.makedirs('./raw_data', exist_ok=True)

print('Generating synthetic genomics dataset...')

Generating synthetic genomics dataset...


In [3]:
# ── Samples ───────────────────────────────────────────────────────────────────
N_SAMPLES = 500
N_PATIENTS = 300

TISSUE_TYPES = ['Tumor', 'Normal', 'Blood', 'PBMC', 'cfDNA']
SEQ_PLATFORMS = ['Illumina_NovaSeq', 'Illumina_HiSeq', 'PacBio_Sequel', 'Oxford_Nanopore']
LIBRARY_PREPS = ['WGS', 'WES', 'RNA-seq', 'Targeted_Panel']
PROJECTS = [f'PROJ_{i:03d}' for i in range(1, 9)]

samples = []
sample_collection_dates = {}
for i in range(1, N_SAMPLES + 1):
    col_date = fake.date_between(start_date=date(2020, 1, 1), end_date=date(2023, 12, 31))
    sample_collection_dates[i] = col_date
    samples.append({
        'sample_id':           i,
        'patient_id':          random.randint(1, N_PATIENTS),
        'tissue_type':         random.choice(TISSUE_TYPES),
        'sequencing_platform': random.choice(SEQ_PLATFORMS),
        'library_prep':        random.choice(LIBRARY_PREPS),
        'collection_date':     col_date.isoformat(),
        'mean_coverage':       round(random.uniform(20.0, 200.0), 1),
        'pct_bases_20x':       round(random.uniform(0.70, 0.99), 4),
        'project_id':          random.choice(PROJECTS),
    })

df_samples = pd.DataFrame(samples)
df_samples.to_csv('./raw_data/samples.csv', index=False)
print(f'samples.csv        — {len(df_samples):,} rows')
df_samples.head(3)

samples.csv        — 500 rows


,sample_id,patient_id,tissue_type,sequencing_platform,library_prep,collection_date,mean_coverage,pct_bases_20x,project_id
0,1,58,Tumor,PacBio_Sequel,WES,2022-07-22,60.2,0.9136,PROJ_002
1,2,217,Tumor,Illumina_NovaSeq,WGS,2020-02-06,59.4,0.8466,PROJ_001
2,3,288,Normal,Oxford_Nanopore,WES,2021-02-05,100.9,0.7807,PROJ_001


In [4]:
# ── Genes ─────────────────────────────────────────────────────────────────────
N_GENES = 100

# Real cancer gene names (first ~20)
REAL_CANCER_GENES = [
    'TP53', 'BRCA1', 'BRCA2', 'KRAS', 'EGFR', 'PIK3CA', 'PTEN', 'APC',
    'RB1', 'MYC', 'CDH1', 'VHL', 'MLH1', 'MSH2', 'ATM', 'PALB2',
    'CDKN2A', 'SMAD4', 'STK11', 'NF1',
]

# Synthetic gene names using biological naming patterns
import string
synthetic_genes = []
prefixes = ['FGFR', 'MAP2K', 'RAF', 'RAS', 'CDK', 'ERBB', 'MET', 'ALK',
            'RET', 'KIT', 'PDGFR', 'VEGFR', 'IGF', 'STAT', 'JAK', 'SRC',
            'ABL', 'BCL', 'MDM', 'FOXO', 'RUNX', 'EZH', 'DNMT', 'TET',
            'IDH', 'ARID', 'CTCF', 'SETD', 'KDM', 'BAP']
used_names = set(REAL_CANCER_GENES)
rng_state = random.getstate()
for pfx in prefixes:
    for suffix in range(1, 5):
        name = f'{pfx}{suffix}'
        if name not in used_names:
            synthetic_genes.append(name)
            used_names.add(name)
        if len(synthetic_genes) >= N_GENES - len(REAL_CANCER_GENES):
            break
    if len(synthetic_genes) >= N_GENES - len(REAL_CANCER_GENES):
        break

ALL_GENE_NAMES = REAL_CANCER_GENES + synthetic_genes[:N_GENES - len(REAL_CANCER_GENES)]
ALL_GENE_NAMES = ALL_GENE_NAMES[:N_GENES]

CHROMOSOMES = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
BIOTYPES = ['protein_coding', 'lncRNA', 'pseudogene', 'miRNA']
BIOTYPE_WEIGHTS = [0.60, 0.25, 0.10, 0.05]

genes = []
gene_info = {}  # gene_id -> {chromosome, start_pos, end_pos}
for i, gname in enumerate(ALL_GENE_NAMES, start=1):
    chrom = random.choice(CHROMOSOMES)
    start = random.randint(1, 240_000_000)
    end   = start + random.randint(1000, 2_500_000)
    gene_info[i] = {'chromosome': chrom, 'start_pos': start, 'end_pos': end}
    genes.append({
        'gene_id':       i,
        'gene_name':     gname,
        'chromosome':    chrom,
        'start_pos':     start,
        'end_pos':       end,
        'strand':        random.choice(['+', '-']),
        'biotype':       random.choices(BIOTYPES, weights=BIOTYPE_WEIGHTS, k=1)[0],
        'is_cancer_gene': (i <= len(REAL_CANCER_GENES)) or (random.random() < 0.10),
    })

df_genes = pd.DataFrame(genes)
df_genes.to_csv('./raw_data/genes.csv', index=False)
print(f'genes.csv          — {len(df_genes):,} rows')
print(f'  Cancer genes: {df_genes["is_cancer_gene"].sum()}')
df_genes.head(3)

genes.csv          — 100 rows
  Cancer genes: 26


,gene_id,gene_name,chromosome,start_pos,end_pos,strand,biotype,is_cancer_gene
0,1,TP53,chr1,61315568,62257439,+,protein_coding,True
1,2,BRCA1,chr1,158294465,160412051,+,protein_coding,True
2,3,BRCA2,chrX,174628104,174785817,-,pseudogene,True


In [5]:
# ── Variant Calls ─────────────────────────────────────────────────────────────
N_CALLS = 2000

CALLERS = ['GATK', 'DeepVariant', 'Strelka2', 'Mutect2']
FILTER_STATUSES = ['PASS', 'PASS', 'PASS', 'LowQuality', 'Contamination', 'Artifact']

sample_ids = df_samples['sample_id'].tolist()

variant_calls = []
call_sample_map = {}   # call_id -> sample_id
call_date_map   = {}   # call_id -> call_date

for i in range(1, N_CALLS + 1):
    sid      = random.choice(sample_ids)
    col_date = sample_collection_dates[sid]
    # call_date must be >= collection_date
    days_after = random.randint(0, 365)
    call_date  = col_date + timedelta(days=days_after)
    call_sample_map[i] = sid
    call_date_map[i]   = call_date
    variant_calls.append({
        'call_id':       i,
        'sample_id':     sid,
        'call_date':     call_date.isoformat(),
        'caller':        random.choice(CALLERS),
        'filter_status': random.choice(FILTER_STATUSES),
    })

df_calls = pd.DataFrame(variant_calls)
df_calls.to_csv('./raw_data/variant_calls.csv', index=False)
print(f'variant_calls.csv  — {len(df_calls):,} rows')
print(f'  Filter status counts:\n{df_calls["filter_status"].value_counts().to_string()}')
df_calls.head(3)

variant_calls.csv  — 2,000 rows
  Filter status counts:
filter_status
PASS             1000
LowQuality        350
Artifact          348
Contamination     302


,call_id,sample_id,call_date,caller,filter_status
0,1,87,2023-12-29,Strelka2,PASS
1,2,403,2022-03-14,Mutect2,PASS
2,3,431,2020-12-05,DeepVariant,Artifact


In [6]:
# ── Variants ──────────────────────────────────────────────────────────────────
N_VARIANTS = 5000

BASES = ['A', 'T', 'C', 'G']
VARIANT_TYPES = ['SNV', 'SNV', 'SNV', 'Insertion', 'Deletion']
CONSEQUENCES = [
    'missense_variant', 'synonymous_variant', 'stop_gained',
    'frameshift_variant', 'splice_site_variant', 'intron_variant'
]

call_ids  = df_calls['call_id'].tolist()
gene_ids  = df_genes['gene_id'].tolist()

variants = []
for i in range(1, N_VARIANTS + 1):
    cid    = random.choice(call_ids)
    gid    = random.choice(gene_ids)
    gdata  = gene_info[gid]
    pos    = random.randint(gdata['start_pos'], gdata['end_pos'])
    ref    = random.choice(BASES)
    alt    = random.choice([b for b in BASES if b != ref])
    variants.append({
        'variant_id':   i,
        'call_id':      cid,
        'gene_id':      gid,
        'chromosome':   gdata['chromosome'],
        'position':     pos,
        'ref_allele':   ref,
        'alt_allele':   alt,
        'variant_type': random.choice(VARIANT_TYPES),
        'consequence':  random.choice(CONSEQUENCES),
        'af_tumor':     round(random.uniform(0.05, 0.95), 4),
        'depth':        random.randint(20, 500),
        'quality_score': round(random.uniform(20.0, 60.0), 2),
    })

df_variants = pd.DataFrame(variants)
df_variants.to_csv('./raw_data/variants.csv', index=False)
print(f'variants.csv       — {len(df_variants):,} rows')
df_variants.head(3)

variants.csv       — 5,000 rows


,variant_id,call_id,gene_id,chromosome,position,ref_allele,alt_allele,variant_type,consequence,af_tumor,depth,quality_score
0,1,388,52,chr10,121745068,C,G,Deletion,frameshift_variant,0.5168,75,47.25
1,2,1324,32,chr20,14654760,T,A,SNV,missense_variant,0.5677,142,22.63
2,3,207,36,chr16,120483867,G,T,Insertion,missense_variant,0.2479,441,55.18


## 6. Dataset Summary

In [7]:
import os

print('=== Raw Data Summary ===')
print(f'  Samples       : {len(df_samples):>6,} rows | {len(df_samples.columns)} columns')
print(f'  Genes         : {len(df_genes):>6,} rows | {len(df_genes.columns)} columns')
print(f'  Variant Calls : {len(df_calls):>6,} rows | {len(df_calls.columns)} columns')
print(f'  Variants      : {len(df_variants):>6,} rows | {len(df_variants.columns)} columns')
print()
print('Files on disk:')
for fname in sorted(os.listdir('./raw_data')):
    fpath = f'./raw_data/{fname}'
    size  = os.path.getsize(fpath)
    print(f'  {fname:30s}  {size/1024:6.1f} KB')

=== Raw Data Summary ===
  Samples       :    500 rows | 9 columns
  Genes         :    100 rows | 8 columns
  Variant Calls :  2,000 rows | 5 columns
  Variants      :  5,000 rows | 12 columns

Files on disk:
  genes.csv                          5.2 KB
  samples.csv                       33.9 KB
  variant_calls.csv                 70.0 KB
  variants.csv                     347.6 KB


In [8]:
# Quick descriptive statistics
print('=== Coverage Distribution (mean_coverage) ===')
print(df_samples['mean_coverage'].describe().round(2))
print()
print('=== Filter Status Distribution ===')
print(df_calls['filter_status'].value_counts())
print()
print('=== Top 5 Projects by Sample Count ===')
print(df_samples['project_id'].value_counts().head())

=== Coverage Distribution (mean_coverage) ===
count    500.00
mean     110.03
std       52.63
min       20.80
25%       65.57
50%      106.35
75%      158.43
max      199.90
Name: mean_coverage, dtype: float64

=== Filter Status Distribution ===
filter_status
PASS             1000
LowQuality        350
Artifact          348
Contamination     302
Name: count, dtype: int64

=== Top 5 Projects by Sample Count ===
project_id
PROJ_003    74
PROJ_002    64
PROJ_006    62
PROJ_005    62
PROJ_008    62
Name: count, dtype: int64


## 7. Data Model (Entity-Relationship Diagram)

```
samples              variant_calls       variants             genes
───────              ─────────────       ────────             ─────
sample_id    ◄─────  sample_id           variant_id           gene_id
patient_id           call_id     ◄─────  call_id     ──────►  gene_name
tissue_type          call_date           gene_id              chromosome
sequencing_platform  caller              chromosome           start_pos
library_prep         filter_status       position             end_pos
collection_date                          ref_allele           strand
mean_coverage                            alt_allele           biotype
pct_bases_20x                            variant_type         is_cancer_gene
project_id                               consequence
                                         af_tumor
                                         depth
                                         quality_score
```

This is a classic **transactional (OLTP) schema**. In the subsequent notebooks we will transform this into:
- A **data lake** (Parquet files organized in Bronze/Silver/Gold zones)
- A **data warehouse** (star schema in DuckDB)
- A **lakehouse** (Delta tables with ACID transactions)

## Exercises

**Exercise 1.1** — Explore the raw data

Using the DataFrames already in memory (`df_samples`, `df_genes`, `df_calls`, `df_variants`), answer:

a) What is the average variant sequencing depth (`depth`)?

b) How many unique samples have at least one variant call with `filter_status = 'PASS'`?

c) What fraction of variant calls have a non-PASS filter status (i.e., `filter_status != 'PASS'`)?

d) Which gene biotype has the highest mean `quality_score` across all variants? (Hint: join `df_variants` to `df_genes` on `gene_id`.)

In [ ]:
# Exercise 1.1 — your answers here

# a) Average variant depth
# avg_depth = ...

# b) Unique samples with at least one PASS variant call
# unique_samples_pass = ...

# c) Fraction of non-PASS variant calls
# non_pass_frac = ...

# d) Biotype with highest mean quality_score
# top_biotype = ...

**Exercise 1.2** — Schema design

If you were designing a data warehouse star schema for this genomics dataset, what would your fact and dimension tables look like? Sketch it out in the cell below (as markdown or comments).

Hint: think about what analysts query most often — variant counts, allele frequencies, gene annotations, sample quality. The finest-grain fact table is at the variant level. Possible dimensions: samples, genes, date, call method.

*Your schema design here...*

**Exercise 1.3** — Architecture selection

For each scenario below, which architecture (warehouse, lake, or lakehouse) would you recommend, and why?

a) A hospital wants to generate monthly reports on somatic mutation burden per patient using structured VCF-derived data. Queries are pre-defined and run on a schedule.

b) A sequencing lab receives raw FASTQ files from 200 runs per week. Researchers want to explore them with custom Python scripts and occasionally run bioinformatics tools directly on the files.

c) A biotech company wants to run both SQL dashboards for variant counts and ML models for pathogenicity scoring on the same underlying dataset.

*Your answers here...*

a) ...

b) ...

c) ...

---

**Next:** `02_data_lake.ipynb` — Building a local data lake with PyArrow and Parquet